In [1]:
import pandas as pd

train_df = pd.read_csv("../data/raw/banking77/train.csv")
test_df = pd.read_csv("../data/raw/banking77/test.csv")

X_train = train_df["text"].tolist()
y_train = train_df["category"].tolist()

X_test = test_df["text"].tolist()
y_test = test_df["category"].tolist()

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

Training samples: 10003
Test samples: 3080


In [2]:
categories = sorted(train_df["category"].unique())

label_to_id = {
    label: idx
    for idx, label in enumerate(categories)
}

id_to_label = {
    idx: label
    for label, idx in label_to_id.items()
}

print("Number of classes:", len(categories))
print("\nFirst 10 labels:")

for label, idx in list(label_to_id.items())[:10]:
    print(idx, "→", label)

Number of classes: 77

First 10 labels:
0 → Refund_not_showing_up
1 → activate_my_card
2 → age_limit
3 → apple_pay_or_google_pay
4 → atm_support
5 → automatic_top_up
6 → balance_not_updated_after_bank_transfer
7 → balance_not_updated_after_cheque_or_cash_deposit
8 → beneficiary_not_allowed
9 → cancel_transfer


In [3]:
y_train_ids = [label_to_id[label] for label in y_train]
y_test_ids = [label_to_id[label] for label in y_test]

print(y_train_ids[:10])

[12, 12, 12, 12, 12, 12, 12, 12, 12, 12]


In [4]:
import re

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

In [5]:
text = "I forgot my card PIN!"

print(tokenize(text))

['i', 'forgot', 'my', 'card', 'pin']


In [6]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

word_to_id = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for text in X_train:
    for token in tokenize(text):
        if token not in word_to_id:
            word_to_id[token] = len(word_to_id)

id_to_word = {
    idx: word
    for word, idx in word_to_id.items()
}

print("Vocabulary size:", len(word_to_id))
print("First 20 words:")

for idx, word in list(id_to_word.items())[:20]:
    print(idx, "→", word)

Vocabulary size: 2343
First 20 words:
0 → <PAD>
1 → <UNK>
2 → i
3 → am
4 → still
5 → waiting
6 → on
7 → my
8 → card
9 → what
10 → can
11 → do
12 → if
13 → hasn
14 → t
15 → arrived
16 → after
17 → 2
18 → weeks
19 → have


In [7]:
def text_to_sequence(text):
    return [
        word_to_id.get(token, word_to_id[UNK_TOKEN])
        for token in tokenize(text)
    ]

In [8]:
sample = "I forgot my card PIN"

print(text_to_sequence(sample))

[2, 962, 7, 8, 817]


In [9]:
sequence_lengths = [
    len(text_to_sequence(text))
    for text in X_train
]

print("Minimum length:", min(sequence_lengths))
print("Maximum length:", max(sequence_lengths))
print("Average length:", sum(sequence_lengths) / len(sequence_lengths))

Minimum length: 2
Maximum length: 84
Average length: 12.253923822853144


In [10]:
MAX_LENGTH = 30

In [11]:
MAX_LENGTH = 30

num_truncated = sum(
    length > MAX_LENGTH
    for length in sequence_lengths
)

print("Total training samples:", len(sequence_lengths))
print("Samples longer than MAX_LENGTH:", num_truncated)
print(
    "Percentage truncated:",
    f"{num_truncated / len(sequence_lengths) * 100:.2f}%"
)

Total training samples: 10003
Samples longer than MAX_LENGTH: 489
Percentage truncated: 4.89%


In [12]:
def pad_sequence(sequence, max_length=MAX_LENGTH):
    if len(sequence) > max_length:
        return sequence[:max_length]

    return sequence + [word_to_id[PAD_TOKEN]] * (
        max_length - len(sequence)
    )

In [13]:
sample_sequence = text_to_sequence("I forgot my card PIN")

padded = pad_sequence(sample_sequence)

print("Original:", sample_sequence)
print("Padded:", padded)
print("Length:", len(padded))

Original: [2, 962, 7, 8, 817]
Padded: [2, 962, 7, 8, 817, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Length: 30


In [14]:
X_train_sequences = [
    pad_sequence(text_to_sequence(text))
    for text in X_train
]

X_test_sequences = [
    pad_sequence(text_to_sequence(text))
    for text in X_test
]

print("Training shape:", len(X_train_sequences), len(X_train_sequences[0]))
print("Testing shape:", len(X_test_sequences), len(X_test_sequences[0]))

Training shape: 10003 30
Testing shape: 3080 30


In [15]:
from sklearn.model_selection import train_test_split

X_train_seq, X_val_seq, y_train_id, y_val_id = train_test_split(
    X_train_sequences,
    y_train_ids,
    test_size=0.2,
    random_state=42,
    stratify=y_train_ids
)

print("Training samples:", len(X_train_seq))
print("Validation samples:", len(X_val_seq))
print("Test samples:", len(X_test_sequences))

Training samples: 8002
Validation samples: 2001
Test samples: 3080


In [16]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert to tensors
X_train_tensor = torch.tensor(X_train_seq, dtype=torch.long)
y_train_tensor = torch.tensor(y_train_id, dtype=torch.long)

X_val_tensor = torch.tensor(X_val_seq, dtype=torch.long)
y_val_tensor = torch.tensor(y_val_id, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_sequences, dtype=torch.long)
y_test_tensor = torch.tensor(y_test_ids, dtype=torch.long)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 126
Validation batches: 32
Test batches: 49


In [17]:
X_batch, y_batch = next(iter(train_loader))

print("Input shape:", X_batch.shape)
print("Label shape:", y_batch.shape)
print("Input dtype:", X_batch.dtype)
print("Label dtype:", y_batch.dtype)

Input shape: torch.Size([64, 30])
Label shape: torch.Size([64])
Input dtype: torch.int64
Label dtype: torch.int64


In [18]:
import torch
import torch.nn as nn

VOCAB_SIZE = len(word_to_id)
NUM_CLASSES = len(categories)
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
PAD_ID = word_to_id[PAD_TOKEN]


class TicketClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_classes,
        pad_id
    ):
        super().__init__()

        self.pad_id = pad_id

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_id
        )

        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):

        # x shape: [batch_size, sequence_length]
        embeddings = self.embedding(x)

        # Ignore PAD tokens
        mask = (x != self.pad_id).unsqueeze(-1)

        masked_embeddings = embeddings * mask

        # Number of real tokens in each ticket
        token_counts = mask.sum(dim=1).clamp(min=1)

        # Average only real token embeddings
        pooled = masked_embeddings.sum(dim=1) / token_counts

        # Classification
        return self.classifier(pooled)


model = TicketClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    pad_id=PAD_ID
)

print(model)

TicketClassifier(
  (embedding): Embedding(2343, 64, padding_idx=0)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=77, bias=True)
  )
)


In [19]:
X_batch, y_batch = next(iter(train_loader))

with torch.no_grad():
    outputs = model(X_batch)

print("Input shape:", X_batch.shape)
print("Output shape:", outputs.shape)

Input shape: torch.Size([64, 30])
Output shape: torch.Size([64, 77])


In [20]:
import torch.optim as optim

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [21]:
EPOCHS = 10

for epoch in range(EPOCHS):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    total_train_loss = 0

    for X_batch, y_batch in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(X_batch)

        # Calculate loss
        loss = criterion(outputs, y_batch)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_train_loss += loss.item()

    average_train_loss = total_train_loss / len(train_loader)

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            outputs = model(X_batch)

            predictions = torch.argmax(outputs, dim=1)

            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    validation_accuracy = correct / total

    print(
        f"Epoch {epoch + 1:2d}/{EPOCHS} "
        f"| Train Loss: {average_train_loss:.4f} "
        f"| Validation Accuracy: {validation_accuracy:.4f}"
    )

Epoch  1/10 | Train Loss: 4.1476 | Validation Accuracy: 0.2574
Epoch  2/10 | Train Loss: 3.2790 | Validation Accuracy: 0.4428
Epoch  3/10 | Train Loss: 2.3992 | Validation Accuracy: 0.5497
Epoch  4/10 | Train Loss: 1.9021 | Validation Accuracy: 0.6152
Epoch  5/10 | Train Loss: 1.5656 | Validation Accuracy: 0.6577
Epoch  6/10 | Train Loss: 1.3545 | Validation Accuracy: 0.6912
Epoch  7/10 | Train Loss: 1.1889 | Validation Accuracy: 0.7166
Epoch  8/10 | Train Loss: 1.0704 | Validation Accuracy: 0.7371
Epoch  9/10 | Train Loss: 0.9655 | Validation Accuracy: 0.7536
Epoch 10/10 | Train Loss: 0.8623 | Validation Accuracy: 0.7651


In [22]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

all_val_predictions = []
all_val_labels = []

with torch.no_grad():

    for X_batch, y_batch in val_loader:

        outputs = model(X_batch)
        predictions = torch.argmax(outputs, dim=1)

        all_val_predictions.extend(predictions.tolist())
        all_val_labels.extend(y_batch.tolist())

val_accuracy = accuracy_score(
    all_val_labels,
    all_val_predictions
)

print(f"Validation Accuracy: {val_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        all_val_labels,
        all_val_predictions,
        target_names=categories
    )
)

Validation Accuracy: 0.7651

Classification Report:
                                                  precision    recall  f1-score   support

                           Refund_not_showing_up       0.93      0.81      0.87        32
                                activate_my_card       0.67      0.88      0.76        32
                                       age_limit       0.95      0.86      0.90        22
                         apple_pay_or_google_pay       1.00      0.92      0.96        25
                                     atm_support       0.88      0.88      0.88        17
                                automatic_top_up       0.84      0.84      0.84        25
         balance_not_updated_after_bank_transfer       0.74      0.50      0.60        34
balance_not_updated_after_cheque_or_cash_deposit       0.78      0.89      0.83        36
                         beneficiary_not_allowed       0.66      0.61      0.63        31
                                 cancel_transfe

In [23]:
# Convert validation sequences back to text.
# We use the original training texts identified by the split indices,
# so first we need to create the split with indices if not already available.

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Recreate TF-IDF using training text only
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(train_df["text"])

# Retrain the scikit-learn baseline
model_sklearn = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_sklearn.fit(X_train_tfidf, train_df["category"])

print("TF-IDF vocabulary size:", len(vectorizer.vocabulary_))
print("Baseline model restored.")

TF-IDF vocabulary size: 10292
Baseline model restored.


In [26]:
X_val_tfidf = vectorizer.transform(X_val_text)

sklearn_val_predictions = model_sklearn.predict(X_val_tfidf)

sklearn_val_accuracy = accuracy_score(
    y_val_text,
    sklearn_val_predictions
)

print(f"scikit-learn validation accuracy: {sklearn_val_accuracy:.4f}")
print(f"PyTorch validation accuracy:      {val_accuracy:.4f}")

scikit-learn validation accuracy: 0.9455
PyTorch validation accuracy:      0.7651


In [30]:
ADDITIONAL_EPOCHS = 5

for epoch in range(ADDITIONAL_EPOCHS):

    # Training
    model.train()
    total_train_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    average_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            outputs = model(X_batch)
            predictions = torch.argmax(outputs, dim=1)

            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    validation_accuracy = correct / total

    print(
        f"Epoch {epoch + 11:2d}/20 "
        f"| Train Loss: {average_train_loss:.4f} "
        f"| Validation Accuracy: {validation_accuracy:.4f}"
    )

Epoch 11/20 | Train Loss: 0.2429 | Validation Accuracy: 0.8531
Epoch 12/20 | Train Loss: 0.2296 | Validation Accuracy: 0.8526
Epoch 13/20 | Train Loss: 0.2182 | Validation Accuracy: 0.8546
Epoch 14/20 | Train Loss: 0.2044 | Validation Accuracy: 0.8566
Epoch 15/20 | Train Loss: 0.2089 | Validation Accuracy: 0.8511


In [31]:
from pathlib import Path
import torch

model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "vocab_size": VOCAB_SIZE,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_classes": NUM_CLASSES,
    "pad_id": PAD_ID,
    "word_to_id": word_to_id,
    "id_to_label": id_to_label,
    "max_length": MAX_LENGTH,
}

torch.save(
    checkpoint,
    model_dir / "ticket_classifier.pt"
)

print("Model checkpoint saved.")

Model checkpoint saved.


In [32]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

all_test_predictions = []
all_test_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        predictions = outputs.argmax(dim=1)

        all_test_predictions.extend(predictions.tolist())
        all_test_labels.extend(y_batch.tolist())

test_accuracy = accuracy_score(
    all_test_labels,
    all_test_predictions
)

print(f"PyTorch Test Accuracy: {test_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        all_test_labels,
        all_test_predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=[id_to_label[i] for i in range(NUM_CLASSES)],
        zero_division=0
    )
)

PyTorch Test Accuracy: 0.8643

Classification Report:
                                                  precision    recall  f1-score   support

                           Refund_not_showing_up       0.86      0.90      0.88        40
                                activate_my_card       0.95      0.93      0.94        40
                                       age_limit       1.00      0.95      0.97        40
                         apple_pay_or_google_pay       0.95      0.97      0.96        40
                                     atm_support       1.00      0.95      0.97        40
                                automatic_top_up       1.00      0.93      0.96        40
         balance_not_updated_after_bank_transfer       0.65      0.75      0.70        40
balance_not_updated_after_cheque_or_cash_deposit       0.82      0.90      0.86        40
                         beneficiary_not_allowed       0.75      0.82      0.79        40
                                 cancel_trans

In [33]:
from pathlib import Path
import json
import torch

model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

# Save model weights and configuration
checkpoint = {
    "model_state_dict": model.state_dict(),
    "vocab_size": VOCAB_SIZE,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_classes": NUM_CLASSES,
    "pad_id": PAD_ID,
    "max_length": MAX_LENGTH,
}

torch.save(
    checkpoint,
    model_dir / "ticket_classifier.pt"
)

# Save vocabulary
with open(model_dir / "word_to_id.json", "w", encoding="utf-8") as f:
    json.dump(word_to_id, f, ensure_ascii=False, indent=2)

# Save label mapping
with open(model_dir / "id_to_label.json", "w", encoding="utf-8") as f:
    json.dump(id_to_label, f, ensure_ascii=False, indent=2)

print("Model and preprocessing assets saved.")

Model and preprocessing assets saved.


In [34]:
for path in sorted(model_dir.iterdir()):
    if path.is_file():
        print(f"{path.name}: {path.stat().st_size:,} bytes")

id_to_label.json: 2,566 bytes
ticket_classifier.pt: 675,911 bytes
word_to_id.json: 44,729 bytes


In [35]:
import json
import torch

# Load preprocessing assets
with open(model_dir / "word_to_id.json", encoding="utf-8") as f:
    loaded_word_to_id = json.load(f)

with open(model_dir / "id_to_label.json", encoding="utf-8") as f:
    loaded_id_to_label = {
        int(key): value
        for key, value in json.load(f).items()
    }

# Load checkpoint
checkpoint = torch.load(
    model_dir / "ticket_classifier.pt",
    map_location="cpu",
    weights_only=True
)

# Rebuild model architecture
loaded_model = TicketClassifier(
    vocab_size=checkpoint["vocab_size"],
    embedding_dim=checkpoint["embedding_dim"],
    hidden_dim=checkpoint["hidden_dim"],
    num_classes=checkpoint["num_classes"],
    pad_id=checkpoint["pad_id"]
)

loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()

print("Saved model loaded successfully.")

Saved model loaded successfully.


In [36]:
def predict_intent(text):
    tokens = tokenize(text)

    sequence = [
        loaded_word_to_id.get(token, loaded_word_to_id["<UNK>"])
        for token in tokens
    ]

    max_length = checkpoint["max_length"]
    pad_id = checkpoint["pad_id"]

    sequence = sequence[:max_length]
    sequence += [pad_id] * (max_length - len(sequence))

    input_tensor = torch.tensor([sequence], dtype=torch.long)

    with torch.no_grad():
        logits = loaded_model(input_tensor)
        probabilities = torch.softmax(logits, dim=1)
        predicted_id = logits.argmax(dim=1).item()
        confidence = probabilities[0, predicted_id].item()

    return {
        "intent": loaded_id_to_label[predicted_id],
        "confidence": round(confidence, 4)
    }


predict_intent("My bank card is not working")

{'intent': 'card_not_working', 'confidence': 0.997}

In [37]:
from src.preprocessing.text_preprocessor import preprocess_text

sample = "My bank card is not working"

sequence = preprocess_text(
    sample,
    loaded_word_to_id,
    checkpoint["max_length"],
    checkpoint["pad_id"]
)

print("Sequence length:", len(sequence))
print("First 10 token IDs:", sequence[:10])

ModuleNotFoundError: No module named 'src'

In [38]:
import os

print("Current directory:", os.getcwd())

Current directory: D:\projects\ticketmindai\notebooks


In [39]:
os.chdir(r"D:\projects\ticketmindai")

print("Updated directory:", os.getcwd())

Updated directory: D:\projects\ticketmindai


In [40]:
from src.preprocessing.text_preprocessor import preprocess_text

sample = "My bank card is not working"

sequence = preprocess_text(
    sample,
    loaded_word_to_id,
    checkpoint["max_length"],
    checkpoint["pad_id"]
)

print("Sequence length:", len(sequence))
print("First 10 token IDs:", sequence[:10])

Sequence length: 30
First 10 token IDs: [7, 159, 8, 24, 48, 561, 0, 0, 0, 0]


In [41]:
from src.models.ticket_classifier import TicketClassifier

print("TicketClassifier imported successfully.")

TicketClassifier imported successfully.


In [42]:
from src.inference import TicketIntentPredictor

predictor = TicketIntentPredictor()

result = predictor.predict(
    "My bank card is not working"
)

print(result)

{'intent': 'card_not_working', 'confidence': 0.997}


In [43]:
# Inspect account access and account termination examples

keywords = ["login", "log in", "sign in", "access my account"]

for category in sorted(train_df["category"].unique()):
    examples = train_df[train_df["category"] == category]["text"]

    matches = examples[
        examples.str.lower().str.contains(
            "|".join(keywords), regex=True, na=False
        )
    ]

    if not matches.empty:
        print(f"\nCategory: {category}")
        print(f"Matching examples: {len(matches)}")

        for text in matches.head(10):
            print(f"- {text}")


Category: beneficiary_not_allowed
Matching examples: 1
- Why can't my beneficiary access my account?

Category: card_linking
Matching examples: 1
- I have found my lost or stolen card, would I still have to activate the new card that was sent to me through the mail in order to access my account?

Category: compromised_card
Matching examples: 1
- I just got an email in confirming my purchase with my card. I login to the mobile app on the card and found out there are several purchase were made without me aware of it. Evidently someone had stole my card information and made these fraudulent purchase. I need the transaction to be reverse and have my card suspended.

Category: lost_or_stolen_phone
Matching examples: 2
- I lost my phone and need help so someone doesn't access my account.
- How can I access my account when I have lost my phone?

Category: passcode_forgotten
Matching examples: 6
- I can't log in with this passocde
- I don't remember my login code
- I need a new login code
- I

In [44]:
termination_examples = train_df[
    train_df["category"] == "terminate_account"
]["text"]

print(f"Total examples: {len(termination_examples)}")

for text in termination_examples.head(20):
    print("-", text)

Total examples: 108
- How do I deactivate my account?
- Remove my account immediately.
- Are you able to delete my account?
- How do I discontinue my account?
- please delete account
- I need you to delete my account now.
- I would like to delete my account now, please.
- I would like to invalidate my account.  I am very disappointed with your services.
- I need to delete my account.
- Can you delete the account?
- for god sake, just delete my account i am sick of this
- I need to delete my account, please.
- unhappy with service need to delete account
- This company is bad, delete my account now.
- How do I shut down my account and get out of this company?
- How do I close my account?
- I want to shut down my account as soon as possible.
- cancel my account
- Help me close my account.
- I want to close my account due to poor customer service.


In [45]:
from src.inference import TicketIntentPredictor

predictor = TicketIntentPredictor()

test_phrases = [
    "I am unable to login",
    "I am unable to login into my account",
    "I can't log in",
    "I cannot access my account",
    "I cannot sign in to my banking app",
    "I forgot my passcode",
    "I want to close my account",
    "Please terminate my account",
]

for text in test_phrases:
    result = predictor.predict(text)
    print(
        f"Text: {text}\n"
        f"Intent: {result['intent']} | "
        f"Confidence: {result['confidence']:.2%}\n"
    )

Text: I am unable to login
Intent: declined_cash_withdrawal | Confidence: 66.13%

Text: I am unable to login into my account
Intent: terminate_account | Confidence: 96.65%

Text: I can't log in
Intent: passcode_forgotten | Confidence: 99.44%

Text: I cannot access my account
Intent: terminate_account | Confidence: 55.97%

Text: I cannot sign in to my banking app
Intent: passcode_forgotten | Confidence: 81.24%

Text: I forgot my passcode
Intent: passcode_forgotten | Confidence: 100.00%

Text: I want to close my account
Intent: terminate_account | Confidence: 100.00%

Text: Please terminate my account
Intent: terminate_account | Confidence: 99.97%



In [47]:
# Compare training examples for the relevant intents

categories_to_inspect = [
    "passcode_forgotten",
    "unable_to_verify_identity",
    "terminate_account",
    "cash_withdrawl",
    "cash_withdrawal"
]

for category in categories_to_inspect:
    examples = train_df.loc[
        train_df["category"] == category, "text"
    ]

    print(f"\n{'=' * 60}")
    print(f"Category: {category} | Examples: {len(examples)}")

    for text in examples.head(15):
        print("-", text)


Category: passcode_forgotten | Examples: 105
- Help me!  I don't know what my password is.
- I thought I knew my password but I guess I was wrong, what can I do now?
- I am unable to access my app due to forgetting my passcode.
- I can't recall my password.
- What should I do if I don't know my password?
- Need a new passcode.
- I can't log in with this passocde
- I forgot my passcode. Now what?
- What are the steps to reset my passcode?
- Where do I go to reset the passcode?
- I am unaware of what my password is.
- I can't remember my passcode.
- I forgot my code
- Help me.  The passcode doesn't work.
- The passcode isn't working for me.

Category: unable_to_verify_identity | Examples: 102
- Can you help me with proving my identity?
- What proof do you need for my identification?
- Are there any reasons that my identity wouldn't be accepted even though everything I've provided is factual?
- I am having some difficulty verifying my id.
- I am not able to verify my id. Why?
- Help me v

In [48]:
# Show the exact category names related to login and withdrawal

for category in sorted(train_df["category"].unique()):
    if any(
        keyword in category
        for keyword in ["login", "passcode", "withdraw", "account", "verify"]
    ):
        count = (train_df["category"] == category).sum()
        print(f"{category}: {count}")

cash_withdrawal_charge: 177
cash_withdrawal_not_recognised: 160
declined_cash_withdrawal: 173
passcode_forgotten: 105
pending_cash_withdrawal: 143
terminate_account: 108
transfer_into_account: 113
unable_to_verify_identity: 102
verify_my_identity: 104
verify_source_of_funds: 113
verify_top_up: 126
why_verify_identity: 121
wrong_exchange_rate_for_cash_withdrawal: 163


In [51]:

import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, classification_report

model.eval()

val_loader = DataLoader(
    TensorDataset(X_val_tensor, y_val_tensor),
    batch_size=64,
    shuffle=False
)

all_predictions = []
all_actual = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        logits = model(inputs)

        predictions = logits.argmax(dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_actual.extend(labels.tolist())

# Build a table of incorrect predictions
errors = pd.DataFrame({
    "actual_id": all_actual,
    "predicted_id": all_predictions
})

errors["actual"] = errors["actual_id"].map(id_to_label)
errors["predicted"] = errors["predicted_id"].map(id_to_label)

wrong = errors[errors["actual_id"] != errors["predicted_id"]]

print(f"Validation examples: {len(errors)}")
print(f"Incorrect predictions: {len(wrong)}")
print(f"Validation accuracy: {(len(errors) - len(wrong)) / len(errors):.2%}")

print("\nMost common misclassification pairs:")
print(
    wrong.groupby(["actual", "predicted"])
    .size()
    .sort_values(ascending=False)
    .head(15)
)

Validation examples: 2001
Incorrect predictions: 298
Validation accuracy: 85.11%

Most common misclassification pairs:
actual                                   predicted                                       
direct_debit_payment_not_recognised      card_payment_not_recognised                         6
exchange_via_app                         fiat_currency_support                               4
failed_transfer                          beneficiary_not_allowed                             4
top_up_failed                            top_up_reverted                                     4
why_verify_identity                      verify_my_identity                                  4
card_delivery_estimate                   card_arrival                                        3
balance_not_updated_after_bank_transfer  pending_transfer                                    3
change_pin                               get_physical_card                                   3
pending_top_up                 

In [50]:
device = torch.device("cpu")
model = model.to(device)

In [52]:

from sklearn.model_selection import train_test_split
import pandas as pd

# Recreate the original 80/20 split from the official training data.
# Keep the official test set untouched.
train_part, val_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["category"]
)

# Additional examples with explicit, defensible labels.
augmentation_data = [
    ("I cannot sign in because I forgot my passcode.", "passcode_forgotten"),
    ("I don't remember the code I use to log in.", "passcode_forgotten"),
    ("I forgot my login password and cannot access the app.", "passcode_forgotten"),
    ("How can I reset my passcode to sign in?", "passcode_forgotten"),
    ("My passcode isn't working when I try to log in.", "passcode_forgotten"),
    ("I need help recovering my forgotten login code.", "passcode_forgotten"),
    ("The app won't let me log in even though my details are correct.",
     "unable_to_verify_identity"),
    ("I cannot sign in because the app cannot verify my identity.",
     "unable_to_verify_identity"),
    ("My identity verification is preventing me from accessing the app.",
     "unable_to_verify_identity"),
]

augmentation_df = pd.DataFrame(
    augmentation_data,
    columns=["text", "category"]
)

# Add synthetic examples to training data only.
train_augmented = pd.concat(
    [train_part, augmentation_df],
    ignore_index=True
)

print("Original training examples:", len(train_part))
print("Added examples:", len(augmentation_df))
print("Augmented training examples:", len(train_augmented))
print("Validation examples:", len(val_part))

print("\nAdded examples by category:")
print(augmentation_df["category"].value_counts())

Original training examples: 8002
Added examples: 9
Augmented training examples: 8011
Validation examples: 2001

Added examples by category:
category
passcode_forgotten           6
unable_to_verify_identity    3
Name: count, dtype: int64


In [53]:

from collections import Counter
import torch
from torch.utils.data import DataLoader, TensorDataset

# Build vocabulary from augmented training data only
token_counts = Counter()

for text in train_augmented["text"]:
    token_counts.update(tokenize(text))

word_to_id_new = {"<PAD>": 0, "<UNK>": 1}

for word, _ in token_counts.most_common():
    if word not in word_to_id_new:
        word_to_id_new[word] = len(word_to_id_new)

categories_new = sorted(train_df["category"].unique())
label_to_id_new = {label: idx for idx, label in enumerate(categories_new)}
id_to_label_new = {idx: label for label, idx in label_to_id_new.items()}

def encode_texts(texts, vocabulary, max_length):
    sequences = []

    for text in texts:
        tokens = tokenize(text)
        ids = [vocabulary.get(token, vocabulary["<UNK>"]) for token in tokens]
        ids = ids[:max_length]
        ids += [vocabulary["<PAD>"]] * (max_length - len(ids))
        sequences.append(ids)

    return torch.tensor(sequences, dtype=torch.long)

X_train_new = encode_texts(
    train_augmented["text"], word_to_id_new, MAX_LENGTH
)
y_train_new = torch.tensor(
    [label_to_id_new[label] for label in train_augmented["category"]],
    dtype=torch.long
)

X_val_new = encode_texts(
    val_part["text"], word_to_id_new, MAX_LENGTH
)
y_val_new = torch.tensor(
    [label_to_id_new[label] for label in val_part["category"]],
    dtype=torch.long
)

train_loader_new = DataLoader(
    TensorDataset(X_train_new, y_train_new),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader_new = DataLoader(
    TensorDataset(X_val_new, y_val_new),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Vocabulary size:", len(word_to_id_new))
print("Training examples:", len(X_train_new))
print("Validation examples:", len(X_val_new))
print("Number of classes:", len(categories_new))

Vocabulary size: 2151
Training examples: 8011
Validation examples: 2001
Number of classes: 77


In [54]:

import torch.nn as nn
import torch.optim as optim

device = torch.device("cpu")

EMBEDDING_DIM = 64
HIDDEN_DIM = 128
PAD_ID_NEW = word_to_id_new["<PAD>"]

model_new = TicketClassifier(
    vocab_size=len(word_to_id_new),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=len(categories_new),
    pad_id=PAD_ID_NEW
).to(device)

criterion_new = nn.CrossEntropyLoss()
optimizer_new = optim.Adam(model_new.parameters(), lr=0.001)

EPOCHS_NEW = 15
best_val_accuracy = 0.0
best_state = None

for epoch in range(EPOCHS_NEW):
    model_new.train()
    total_loss = 0.0

    for inputs, labels in train_loader_new:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer_new.zero_grad()
        logits = model_new(inputs)
        loss = criterion_new(logits, labels)
        loss.backward()
        optimizer_new.step()

        total_loss += loss.item()

    model_new.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader_new:
            inputs, labels = inputs.to(device), labels.to(device)
            predictions = model_new(inputs).argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    val_accuracy = correct / total

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model_new.state_dict().items()
        }

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS_NEW} | "
        f"Loss: {total_loss / len(train_loader_new):.4f} | "
        f"Validation accuracy: {val_accuracy:.2%}"
    )

model_new.load_state_dict(best_state)
print(f"\nBest validation accuracy: {best_val_accuracy:.2%}")

Epoch 01/15 | Loss: 4.1178 | Validation accuracy: 24.54%
Epoch 02/15 | Loss: 3.1625 | Validation accuracy: 45.98%
Epoch 03/15 | Loss: 2.3008 | Validation accuracy: 56.67%
Epoch 04/15 | Loss: 1.8087 | Validation accuracy: 62.82%
Epoch 05/15 | Loss: 1.5176 | Validation accuracy: 66.52%
Epoch 06/15 | Loss: 1.2920 | Validation accuracy: 69.87%
Epoch 07/15 | Loss: 1.1399 | Validation accuracy: 72.81%
Epoch 08/15 | Loss: 1.0166 | Validation accuracy: 73.81%
Epoch 09/15 | Loss: 0.9224 | Validation accuracy: 75.96%
Epoch 10/15 | Loss: 0.8407 | Validation accuracy: 76.61%
Epoch 11/15 | Loss: 0.7621 | Validation accuracy: 76.96%
Epoch 12/15 | Loss: 0.7135 | Validation accuracy: 78.51%
Epoch 13/15 | Loss: 0.6454 | Validation accuracy: 78.91%
Epoch 14/15 | Loss: 0.6000 | Validation accuracy: 79.46%
Epoch 15/15 | Loss: 0.5564 | Validation accuracy: 80.51%

Best validation accuracy: 80.51%


In [55]:

def predict_with_model(text, model, vocabulary, id_to_label, max_length):
    model.eval()

    tokens = tokenize(text)
    ids = [vocabulary.get(token, vocabulary["<UNK>"]) for token in tokens]
    ids = ids[:max_length]
    ids += [vocabulary["<PAD>"]] * (max_length - len(ids))

    input_tensor = torch.tensor([ids], dtype=torch.long)

    with torch.no_grad():
        logits = model(input_tensor)
        probabilities = torch.softmax(logits, dim=1)
        predicted_id = logits.argmax(dim=1).item()

    return {
        "intent": id_to_label[predicted_id],
        "confidence": round(probabilities[0, predicted_id].item(), 4)
    }


phrases = [
    "I am unable to login",
    "I am unable to login into my account",
    "I can't log in",
    "I cannot access my account",
    "I cannot sign in to my banking app",
    "I forgot my passcode",
    "I want to close my account"
]

for phrase in phrases:
    result = predict_with_model(
        phrase, model_new, word_to_id_new,
        id_to_label_new, MAX_LENGTH
    )
    print(f"{phrase}\n  {result}\n")

I am unable to login
  {'intent': 'beneficiary_not_allowed', 'confidence': 0.468}

I am unable to login into my account
  {'intent': 'terminate_account', 'confidence': 0.4103}

I can't log in
  {'intent': 'passcode_forgotten', 'confidence': 0.895}

I cannot access my account
  {'intent': 'terminate_account', 'confidence': 0.836}

I cannot sign in to my banking app
  {'intent': 'card_linking', 'confidence': 0.3713}

I forgot my passcode
  {'intent': 'passcode_forgotten', 'confidence': 0.9999}

I want to close my account
  {'intent': 'terminate_account', 'confidence': 0.9994}

